# 第 4 章 · LangGraph 核心：图与状态

> 前三章我们都在用"开箱组件"。本章打开引擎盖，直接驾驶 **LangGraph**——LangChain 生态的编排引擎，也是 `create_agent` 的底层。
>
> 学完本章，你将理解 LangGraph 的三个根本概念：**State（状态）、Node（节点）、Edge（边）**，并能手工搭建一个 ReAct Agent——做完你会发现：`create_agent` 再无秘密可言。

---

## 1. 为什么 LCEL 不够用？

第 2 章说过：LCEL 是**流水线（DAG）**——数据只能向前流。但 Agent 需要：

| 需求 | 举例 | 图论语言 |
|---|---|---|
| 循环 | "评审不满意 → 回去重写" | **回边**（cycle） |
| 条件跳转 | "情绪负面 → 转人工；正面 → 自动回复" | **条件边** |
| 状态累积 | 多轮对话历史、中间产物 | **全局 State + Reducer** |
| 暂停与恢复 | 等人工审批后继续 | **可持久化的执行现场** |

LangGraph 的回答：**把计算建模为状态机**——

```mermaid
flowchart LR
    S0(("State v0")) --> N1["Node A<br/>f(state) → 部分更新"]
    N1 --> S1(("State v1"))
    S1 --> N2["Node B"]
    N2 --> S2(("State v2"))
    S2 -.->|"条件边：可能回到 A（循环）"| N1
    S2 --> E(("END"))
    style S1 fill:#e6f4ea,stroke:#34a853
    style S2 fill:#e6f4ea,stroke:#34a853
```

| 概念 | 定义 | 类比 |
|---|---|---|
| **State** | 一个 TypedDict，图执行期间共享的数据 | 电路板上的电压分布 |
| **Node** | 普通 Python 函数：接收 State，返回**部分更新** | 电路元件 |
| **Edge** | 节点间的连接（普通边 / 条件边 / 回边） | 导线 |
| **Reducer** | 控制"部分更新"如何合并进 State 的函数 | 元件的接线规则 |
| **compile** | 把图定义编译成可执行的 Runnable | 电路板上电 |

---

## 2. 第一张图：五个步骤

构建图的固定套路：**定义 State → 写节点函数 → 建图加节点 → 连边 → 编译**。


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置环境变量 DEEPSEEK_API_KEY"

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ① 定义 State：图执行期间的共享数据
class GreetingState(TypedDict):
    name: str
    time_of_day: str
    greeting: str

# ② 节点函数：输入 State，返回"部分更新"（dict）
def detect_time(state: GreetingState) -> dict:
    from datetime import datetime
    hour = datetime.now().hour
    period = "早上" if hour < 12 else ("下午" if hour < 18 else "晚上")
    return {"time_of_day": period}          # 只更新自己负责的字段

def make_greeting(state: GreetingState) -> dict:
    return {"greeting": f"{state['time_of_day']}好，{state['name']}！"}

# ③④ 建图：加节点、连边
builder = StateGraph(GreetingState)
builder.add_node("detect_time", detect_time)
builder.add_node("make_greeting", make_greeting)
builder.add_edge(START, "detect_time")       # START → detect_time
builder.add_edge("detect_time", "make_greeting")
builder.add_edge("make_greeting", END)       # → END

# ⑤ 编译成 Runnable
graph = builder.compile()

result = graph.invoke({"name": "小明"})
print(result)


{'name': '小明', 'time_of_day': '晚上', 'greeting': '晚上好，小明！'}


> 🔍 注意：**你只传了 `name`**，`time_of_day` 和 `greeting` 是节点们陆续填进去的。节点返回的 dict 不是"替换" State，而是**合并**——合并规则由 Reducer 决定（下一节）。

### 把图画出来

LangGraph 可以直接导出 Mermaid 图——调试复杂图时极其实用：


In [2]:
print(graph.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	detect_time(detect_time)
	make_greeting(make_greeting)
	__end__([<p>__end__</p>]):::last
	__start__ --> detect_time;
	detect_time --> make_greeting;
	make_greeting --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



---

## 3. Reducer：状态合并的"接线规则"

默认 Reducer 是**覆盖**（新值换旧值）。但很多字段需要**累积**——最典型的是消息列表。用 `Annotated[类型, reducer函数]` 声明：


In [3]:
from typing import Annotated
from operator import add

class DebateState(TypedDict):
    topic: str
    arguments: Annotated[list[str], add]   # ← 关键：列表做"追加"而非"覆盖"

def side_a(state: DebateState) -> dict:
    return {"arguments": [f"正方观点：{state['topic']}能提升效率"]}

def side_b(state: DebateState) -> dict:
    return {"arguments": [f"反方观点：{state['topic']}带来依赖风险"]}

b = StateGraph(DebateState)
b.add_node("side_a", side_a)
b.add_node("side_b", side_b)
b.add_edge(START, "side_a")
b.add_edge("side_a", "side_b")
b.add_edge("side_b", END)
debate = b.compile()

r = debate.invoke({"topic": "AI 编程助手", "arguments": []})
print("累积后的 arguments：")
for a in r["arguments"]:
    print(" •", a)


累积后的 arguments：
 • 正方观点：AI 编程助手能提升效率
 • 反方观点：AI 编程助手带来依赖风险


> 📌 **LangGraph 内置最重要的 Reducer 是 `add_messages`**：它不仅追加消息，还能按消息 id 去重/更新——第 3 章的 Agent 之所以能把 ToolMessage 一路累积下来，靠的就是它。

```python
from langgraph.graph.message import add_messages, MessagesState

class MyState(TypedDict):
    messages: Annotated[list, add_messages]   # 标配写法

# 或直接用预制的 MessagesState（等价于上面这个只有 messages 字段的 State）
```

---

## 4. 条件边：让图"自己做决定"

普通边是"铁轨"，条件边是"道岔"——由一个**路由函数**读取 State、返回下一个节点的名字：

**场景**：用户留言情感分流——负面情绪走"安抚专员"，其他走"标准回复"。


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="deepseek-chat", api_key=os.environ["DEEPSEEK_API_KEY"],
                 base_url="https://api.deepseek.com", temperature=0)

class TicketState(TypedDict):
    message: str
    sentiment: str
    reply: str

def classify(state: TicketState) -> dict:
    r = llm.invoke([HumanMessage(content=f"判断以下留言的情绪，只回答「负面」或「非负面」：{state['message']}")])
    return {"sentiment": r.content.strip()}

def route_by_sentiment(state: TicketState) -> str:
    """路由函数：返回下一个节点的名字。"""
    return "comfort" if "负面" in state["sentiment"] else "standard"

def comfort(state: TicketState) -> dict:
    r = llm.invoke([HumanMessage(content=f"你是客服安抚专员，用共情的语气回复这条负面留言（80字内）：{state['message']}")])
    return {"reply": f"[安抚通道] {r.content}"}

def standard(state: TicketState) -> dict:
    r = llm.invoke([HumanMessage(content=f"你是客服，专业简洁地回复这条留言（80字内）：{state['message']}")])
    return {"reply": f"[标准通道] {r.content}"}

b = StateGraph(TicketState)
b.add_node("classify", classify)
b.add_node("comfort", comfort)
b.add_node("standard", standard)
b.add_edge(START, "classify")
b.add_conditional_edges("classify", route_by_sentiment, {"comfort": "comfort", "standard": "standard"})
b.add_edge("comfort", END)
b.add_edge("standard", END)
ticket_bot = b.compile()

print(ticket_bot.invoke({"message": "你们的产品太难用了，提交三次都失败，气死我了！"})["reply"])
print("─" * 55)
print(ticket_bot.invoke({"message": "请问你们的会员套餐包含哪些内容？"})["reply"])


[安抚通道] 非常理解您的心情，反复提交失败确实太让人恼火了。给您带来这么糟糕的体验，真的非常抱歉。我马上帮您核查问题，请放心，我一定尽力帮您解决。
───────────────────────────────────────────────────────


[安抚通道] 您好，非常理解您想了解会员套餐的具体内容，没能及时为您说明是我们的疏忽，真的抱歉。这就为您详细梳理：套餐包含专属折扣、生日礼遇及积分加速等权益。您看这样解释清楚吗？如有其他需求，我随时为您服务。


```mermaid
flowchart LR
    S((START)) --> C["classify<br/>情绪识别"]
    C -->|"负面"| K["comfort<br/>安抚专员"]
    C -->|"非负面"| N["standard<br/>标准客服"]
    K --> E((END))
    N --> E
    style C fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
```

> 💡 路由函数 + LLM 分类器，就是 ADK 委派模式（`transfer_to_agent`）在 LangGraph 里的手工版——**ADK 内建的"模式"，在 LangGraph 里都能用基础件搭出来，代价是代码量，收益是掌控力**。

---

## 5. 终极实战：手工搭建 ReAct Agent

现在把第 3 章 `create_agent` 的内部亲手实现一遍。配料：

| 配料 | 来源 |
|---|---|
| `MessagesState` | 预置 State（`messages` 字段 + `add_messages` Reducer） |
| `llm.bind_tools(tools)` | 让模型知道有哪些工具可调 |
| `ToolNode` | 预置节点：执行 AIMessage 里请求的 tool_calls |
| `tools_condition` | 预置路由：有 tool_calls → tools；否则 → END |


In [5]:
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> dict:
    """查询城市天气。

    Args:
        city: 城市名，如"杭州"。
    """
    return {"city": city, "weather": "多云转晴", "temp": "24°C"}

# 模型节点：带工具感知地调用 LLM
llm_with_tools = llm.bind_tools([get_weather])

def call_model(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}    # add_messages 会自动追加

# 建图：这就是 create_agent 的最小内核
b = StateGraph(MessagesState)
b.add_node("call_model", call_model)
b.add_node("tools", ToolNode([get_weather]))
b.add_edge(START, "call_model")
b.add_conditional_edges("call_model", tools_condition)   # 有工具调用→tools，否则→END
b.add_edge("tools", "call_model")                        # ← 回边！循环在这里
react_agent = b.compile()

r = react_agent.invoke({"messages": [HumanMessage(content="杭州今天适合户外跑步吗？")]})
for m in r["messages"]:
    if m.type == "ai" and m.tool_calls:
        print(f"🧠 模型决策：调用 {m.tool_calls[0]['name']}{m.tool_calls[0]['args']}")
    elif m.type == "tool":
        print(f"🔧 工具观察：{m.content[:60]}")
    elif m.type == "ai":
        print(f"💬 最终回答：{m.content}")


🧠 模型决策：调用 get_weather{'city': '杭州'}
🔧 工具观察：{"city": "杭州", "weather": "多云转晴", "temp": "24°C"}
💬 最终回答：根据查询到的杭州天气情况，我来为您分析一下今天是否适合户外跑步：

## 杭州今日天气概况
- **天气**：多云转晴
- **气温**：24°C

## 跑步适宜度分析

**✅ 非常适合户外跑步！** 理由如下：

1. **温度适宜**：24°C 是户外跑步的黄金温度区间（最佳跑步温度一般在15-25°C之间），既不会太热导致大量出汗，也不会太冷影响身体状态。

2. **天气良好**：多云转晴，没有降雨，路面干燥，不用担心湿滑问题，跑步体验会很舒适。

3. **阳光适中**：多云天气还能适当遮挡紫外线，减少暴晒，对皮肤更友好。

## 小建议
- **时间选择**：建议选择早晨或傍晚时段跑步，避开正午阳光最强的时候。
- **补水**：虽然温度不算高，但跑步时仍要注意及时补充水分。
- **防晒**：转晴后紫外线会增强，可适当涂抹防晒霜。

总之，今天杭州的天气非常适合户外跑步，祝您跑得愉快！🏃♂️


```mermaid
flowchart LR
    S((START)) --> M["call_model<br/>LLM 决策"]
    M -->|"tools_condition:<br/>有 tool_calls"| T["ToolNode<br/>执行工具"]
    M -->|"无 tool_calls"| E((END))
    T -->|"回边"| M
    style M fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style T fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

> 🎓 **顿悟时刻**：`create_agent` = `MessagesState` + 模型节点 + `ToolNode` + `tools_condition` + 回边（再加上中间件、结构化输出等装饰）。框架没有魔法——**它只是把你现在会写的代码预先写好了**。当你需要跳出预设（自定义审批节点、特殊循环、分叉合并），手工建图的能力就是必需品。

---

## 6. 与 ADK 对照 🔄

| LangGraph | ADK | 备注 |
|---|---|---|
| State（TypedDict） | Session State（dict） | LangGraph 有类型与 Reducer，更严格 |
| Reducer（`add` / `add_messages`） | state_delta 合并（框架托管） | ADK 简单场景零配置 |
| 线性链 START→A→B→END | `SequentialAgent` | 等价 |
| 扇出 + 汇聚 | `ParallelAgent` | LangGraph 需自设汇聚节点 |
| 条件边 + 回边 | `LoopAgent` + escalate | 循环即回边 |
| 路由函数 | LLM 委派（transfer_to_agent） | ADK 内建 LLM 路由 |
| `ToolNode` + `tools_condition` | Runner 的工具执行循环 | create_agent 的内核 |

**一句话总结**：ADK 给你四块预制积木，LangGraph 给你一盒通用零件加图纸——本章之后，你已经能读懂任何基于 LangGraph 的开源 Agent 项目。

---

## 📌 本章要点回顾

- LangGraph 五要素：**State → Node → Edge → Reducer → compile**；
- 节点返回**部分更新**，`Annotated[list, add]` 声明累积式字段；
- `add_conditional_edges` + 路由函数 = LLM 驱动的分支决策；
- `tools_condition` + 回边 = ReAct 循环——`create_agent` 的全部秘密；
- 图可导出 Mermaid 可视化，调试利器。

> ➡️ 下一章：[05-持久化-流式与人机协同](05-持久化-流式与人机协同.ipynb) —— Checkpointer、时间旅行与 interrupt。


---

## 🧪 本章练习

### 1. State 与 Reducer 单元测试（基础）

设计一个包含 `messages`、`attempts`、`artifacts` 和 `final_answer` 的 State，为每个字段选择覆盖、累加或 `add_messages` 等合并策略。直接调用节点并模拟并行更新，写测试证明 Reducer 行为符合预期；至少展示一个选错 Reducer 导致历史丢失或数据重复的反例。

### 2. 条件路由工单图（进阶）

构建“分类 → 处理 → 汇总”的工单图：账单、技术与普通咨询走不同节点，低置信度时进入补问节点。路由函数必须有默认分支，所有路径最终可达 `END`。画出 Mermaid 图，并为每条边准备至少一个测试输入。

### 3. 手工 ReAct 的安全边界（进阶）

在本章手工 ReAct 图上加入最大循环次数、工具允许列表与累计成本预算；超限后不再回到模型节点，而是返回可解释的终止结果。构造一个会反复调用工具的输入，证明图不会无限循环，并比较这一实现与 `create_agent` 的封装取舍。

### 4. 计划—执行—复盘图（工程综合）

实现一个简化的 Plan-and-Execute 图：规划节点生成步骤，执行节点逐步完成，复盘节点决定结束、重规划或人工升级。State 中要保留计划版本、当前步骤、证据和失败原因；设置最大重规划次数。验收时覆盖正常完成、执行失败后重规划、达到上限三条路径。

### 5. 与 ADK 工作流互译（跨框架）

把练习 2 或 4 改写为 ADK 架构草图，使用 `SequentialAgent`、`LoopAgent`、`sub_agents` 或普通 LLM Agent 中最合适的组合。逐项映射 State/Reducer/边，并指出 ADK 框架托管状态合并后，哪些控制能力更简洁、哪些能力更难显式表达。
